# Population-weighted distance-decay distribution
First run [3_population_distribution.ipynb](3_population_distribution.ipynb) through its aggregate export section. For airport demand, also run SD4’s airport export. Run this notebook with the repository root as the working directory.

Workplace-area/residence-region OD targets constrain allocation. Within each pairing, workplace probabilities are proportional to remaining job population times `exp(-distance / DISTANCE_SCALE)`, with origins sampled by remaining residents. Capacities and OD quotas are consumed exactly after rounding/balancing.

In [24]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import geopandas as gpd
import pandas as pd

from src import data_filepaths
from src.utils.loaders import load_boundary_pln_area, load_boundary_pln_region
from src.utils.population_distribution import (
    locate_demand_points, distribute_population, build_demand_data,
)
from src.utils.special_demand import (
    hotel_origins, merge_origin_points, distribute_airport_demand,
)

ALLOW_EXPORT = True

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration
Dense pairs use the large size only if **both** endpoint populations exceed it and their remaining capacities and OD quota can accommodate it. All other groups use the small size. Large must be an integer multiple of small.

Counts are rounded to small-size multiples; the larger resident/job total is reduced proportionally to the smaller using largest remainders. Zero OD weights are soft targets, matching the legacy quota allocator. Same-point commutes are allowed. Driving metrics are straight-line estimates, not road routes.

In [25]:
SMALL_SIZE = 50
LARGE_SIZE = 200
DISTANCE_SCALE = 10_000  # metres, EPSG:3414
RANDOM_SEED = 0
DRIVING_SPEED = 40 / 3.6  # metres per second

OUTPUT_PATH = Path(data_filepaths.OUT_DEMAND_DATA) / 'demand_data_v2.json'

# Special demand confis
INCLUDE_AIRPORT_DEMAND = True
AIRPORT_DISTANCE_SCALE = 10_000  # metres; independent of workforce OD allocation
ORIGIN_MERGE_RADIUS = 200  # metres

In [26]:
demand_gdf = gpd.read_parquet(data_filepaths.AGGREGATED_DEMAND_POINTS)
df_ewarr = pd.read_parquet(data_filepaths.WORKPLACE_RESIDENCE_OD)
pln_area_gdf = load_boundary_pln_area()
pln_region_gdf = load_boundary_pln_region()
points_gdf = locate_demand_points(demand_gdf, pln_area_gdf, pln_region_gdf)

outside = points_gdf[['pln_area_id', 'pln_region_id']].isna().any(axis=1)
print(f'Excluded {outside.sum()} points outside planning boundaries:')
print(points_gdf.loc[outside, ['residents', 'workers']].sum())
points_gdf = points_gdf.loc[~outside].copy()
points_gdf[['residents', 'workers']].sum()

Excluded 1 points outside planning boundaries:
residents    199.929035
workers        0.000000
dtype: float64


residents    3.018854e+06
workers      3.013468e+06
dtype: float64

In [27]:
allocated_points, pops_df, od_people = distribute_population(
    points_gdf, df_ewarr.loc[:, pln_region_gdf.index],
    small_size=SMALL_SIZE, large_size=LARGE_SIZE,
    distance_scale=DISTANCE_SCALE, seed=RANDOM_SEED,
)
print(pd.DataFrame({
    'input': points_gdf[['residents', 'workers']].sum(),
    'allocated': allocated_points[['residents', 'workers']].sum(),
}))
print('Group sizes:', pops_df['size'].value_counts().to_dict())
od_people.head(8)

                  input  allocated
residents  3.018854e+06    3013150
workers    3.013468e+06    3013150
Group sizes: {200: 13162, 50: 7615}


,CR,ER,NER,NR,WR
AM,11300,7600,27850,16700,9900
BD,15300,40950,19300,8500,8400
BK,8400,3450,7950,7550,28500
BL,2950,1750,2100,2750,12450
BM,43450,20950,31400,14800,35350
BP,1500,600,1350,2050,11250
BS,11750,4350,13700,6050,6250
BT,5700,2750,4500,2500,9300


## Add airport passenger demand
Run notebook 3's aggregate export and SD4's airport export first. Existing-residential
weights represent all residents; hotel floorspace is a relative guest-population
proxy. Each pool independently supports the configured terminal budget. Origins
outside planning boundaries are reported and excluded before merging. Existing
workforce allocation and point coordinates remain unchanged.

The final JSON's `residents` and `jobs` count total modeled demand, including airport
passengers. Category, pool, membership, and rounding details are exported separately.


In [ ]:
export_points = allocated_points
export_pops = pops_df

if INCLUDE_AIRPORT_DEMAND:
    residential_origins = gpd.read_parquet(data_filepaths.RESIDENTIAL_POPULATION_ORIGINS)
    hotels = hotel_origins(gpd.read_parquet(data_filepaths.CLASSIFIED_BUILDINGS))
    origins = gpd.GeoDataFrame(
        pd.concat([residential_origins, hotels]), geometry="geometry", crs=3414,
    )
    located_origins = locate_demand_points(origins, pln_area_gdf, pln_region_gdf)
    outside_origins = located_origins[["pln_area_id", "pln_region_id"]].isna().any(axis=1)
    print("Excluded origin weights outside planning boundaries:")
    print(located_origins.loc[outside_origins].groupby("pool_id")["population_weight"].sum())
    origin_points, origin_weights, origin_membership = merge_origin_points(
        allocated_points, located_origins.loc[~outside_origins],
        radius=ORIGIN_MERGE_RADIUS, seed=RANDOM_SEED,
    )
    located_origin_points = locate_demand_points(origin_points, pln_area_gdf, pln_region_gdf)
    if located_origin_points[["pln_area_id", "pln_region_id"]].isna().any().any():
        raise ValueError("Merged origin points outside planning boundaries; inspect origin membership")
    airport_demand = gpd.read_parquet(data_filepaths.AIRPORT_DEMAND)
    located_airports = locate_demand_points(airport_demand, pln_area_gdf, pln_region_gdf)
    if located_airports[["pln_area_id", "pln_region_id"]].isna().any().any():
        raise ValueError("Airport destinations outside planning boundaries; correct their coordinates")
    export_points, airport_pops, airport_report = distribute_airport_demand(
        origin_points, origin_weights, airport_demand,
        small_size=SMALL_SIZE, large_size=LARGE_SIZE,
        distance_scale=AIRPORT_DISTANCE_SCALE, seed=RANDOM_SEED,
    )
    export_pops = pd.concat([pops_df, airport_pops], ignore_index=True)
    print(airport_report.to_string(index=False))
    print(f"Added {airport_pops['size'].sum()} airport passengers in {len(airport_pops)} groups")


In [29]:
demand_data = build_demand_data(export_points, export_pops, driving_speed=DRIVING_SPEED)
assert sum(p['residents'] for p in demand_data['points']) == sum(p['size'] for p in demand_data['pops'])
assert sum(p['jobs'] for p in demand_data['points']) == sum(p['size'] for p in demand_data['pops'])

In [30]:
if ALLOW_EXPORT:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("w", encoding="utf-8") as output:
        json.dump(demand_data, output, allow_nan=False)
    print(
        f"Exported {len(demand_data['points'])} points and {len(demand_data['pops'])} groups to {OUTPUT_PATH}"
    )
    
    if INCLUDE_AIRPORT_DEMAND:
        airport_pops.to_parquet(OUTPUT_PATH.with_name("airport_allocations.parquet"), index=False)
        airport_report.to_parquet(OUTPUT_PATH.with_name("airport_demand_report.parquet"), index=False)
        origin_weights.to_parquet(OUTPUT_PATH.with_name("airport_origin_weights.parquet"), index=False)
        origin_membership.to_parquet(OUTPUT_PATH.with_name("airport_origin_membership.parquet"), index=False)


Exported 4268 points and 21742 groups to out\demand_data\demand_data_v2.json
